# Urban Heat in Santo André — Machine Learning Model

**Question:** which characteristics of a place explain its summer land surface temperature, and how much does each one matter?

**Data:** 7,956 random 30 m pixels in the official urban area of Santo André, exported from Google Earth Engine (`scripts/07_ml_dataset_export.js`), summers 2023–2026.

| Column | Meaning |
|---|---|
| `temp` | **Target.** Land surface temperature (°C), median of summer Landsat 8/9 images, ~10 a.m. |
| `ndvi` | Vegetation index (−1 to 1) |
| `ndbi` | Built-up index (−1 to 1) |
| `albedo` | Share of sunlight reflected by the surface (0 to 1) |
| `elevation` | Elevation (m), NASA SRTM |
| `dist_dense_m` | Distance (m) to the nearest dense-vegetation pixel (NDVI ≥ 0.6) |
| `tree_cover_50m` | Share of tree cover within 50 m (ESA WorldCover 2021) |
| `built_cover_50m` | Share of built-up cover within 50 m (ESA WorldCover 2021) |
| `longitude`, `latitude` | Pixel coordinates (used only for spatial cross-validation) |

**Plan:**
1. Load and explore the data
2. Set up a fair evaluation (spatial cross-validation)
3. Baseline model: linear regression
4. Random forest
5. Which features matter? (permutation importance)
6. How does temperature change with tree cover? (partial dependence)
7. Conclusions and limitations

Cells marked **✏️ YOUR TURN** are for you to write. Read the hints, try first, and only then ask for help.

## 1. Load the data

The Earth Engine export saved `santo_andre_urban_heat_ml_dataset.csv` in your Google Drive. Run the cell below and allow Colab to access your Drive.

*If the file is in a different folder, change `CSV_PATH`. You can find the path in the **Files** panel on the left: open `drive/MyDrive`, right-click the file → **Copy path**.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/santo_andre_urban_heat_ml_dataset.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, KFold, cross_validate
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## 2. Explore the data

Before any model, understand the data: ranges, missing values and strange values.

In [ ]:
df.describe().round(3)

In [ ]:
print('Missing values per column:')
print(df.isna().sum())

In [ ]:
FEATURES = ['ndvi', 'ndbi', 'albedo', 'elevation', 'dist_dense_m',
            'tree_cover_50m', 'built_cover_50m']
TARGET = 'temp'

df[[TARGET] + FEATURES].hist(bins=40, figsize=(14, 9))
plt.tight_layout()
plt.show()

### ✏️ YOUR TURN — Exercise 1: correlations

Calculate the **Pearson correlation** between each feature in `FEATURES` and `temp`, and show them **sorted from the most negative to the most positive**.

*Hints:* `df[FEATURES + [TARGET]].corr()` gives a correlation table. Select the `temp` column, drop the row for `temp` itself, then use `.sort_values()`.

**Questions to answer in the markdown cell below your code:**
1. Which feature has the strongest relationship with temperature?
2. Does `ndvi` give a correlation close to the −0.71 you found in Earth Engine? Why might it be a little different?

In [ ]:
# Exercise 1: correlation of each feature with temperature
correlacoes = df[FEATURES + [TARGET]].corr()      # correlations between all columns
correlacoes_temp = correlacoes[TARGET]            # only the temperature column
correlacoes_temp = correlacoes_temp.drop(TARGET)  # remove temp vs temp (always 1)
correlacoes_temp.sort_values()                    # most negative -> most positive

M

**My answers (Exercise 1):**

1. The strongest relationship is with surrounding built-up cover (+0.78), indicating that areas with more built-up surfaces tend to have higher temperatures. The second strongest is surrounding tree cover (−0.77), showing that areas with more trees tend to have lower temperatures. Tree cover has a stronger relationship with temperature than NDVI (−0.72), since NDVI measures all green vegetation, including grass, while tree cover specifically measures the presence of trees.

2. The results differ slightly because they are based on different samples. Analysis 4 randomly selected 2,000 pixels, while script 07 selected a different sample of 8,000 pixels, of which 7,956 remained after excluding pixels with water or missing data. As a result, the values vary slightly (−0.71 versus −0.72). However, they are very similar, suggesting that the relationship is consistent across different samples.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feature in zip(axes.ravel(), FEATURES):
    ax.scatter(df[feature], df[TARGET], s=2, alpha=0.3)
    ax.set_xlabel(feature)
    ax.set_ylabel('temp (°C)')
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

## 3. A fair evaluation: spatial cross-validation

To know if a model is good, we test it on data it **did not see** during training.

The usual way is to split the points randomly. **But here that is a trap.** Neighbouring pixels are very similar (the same street, the same park). If one pixel is in the training set and its neighbour is in the test set, the model can "cheat" by memorising the area, and the score looks better than it really is. This is called **spatial autocorrelation**.

**Solution:** divide the city into blocks of about 1 km × 1 km and keep **whole blocks** together, either in training or in testing. Then the model is always tested on areas it has never seen. `GroupKFold` does this if we give each point the ID of its block.

In [ ]:
BLOCK_SIZE_DEG = 0.01   # about 1 km in Santo André

block_x = np.floor(df['longitude'] / BLOCK_SIZE_DEG).astype(int)
block_y = np.floor(df['latitude'] / BLOCK_SIZE_DEG).astype(int)
df['block_id'] = block_x.astype(str) + '_' + block_y.astype(str)

print('Number of spatial blocks:', df['block_id'].nunique())

plt.figure(figsize=(6, 6))
codes = df['block_id'].astype('category').cat.codes
plt.scatter(df['longitude'], df['latitude'], c=codes % 20, cmap='tab20', s=2)
plt.title('Spatial blocks (colours repeat)')
plt.xlabel('longitude'); plt.ylabel('latitude')
plt.show()

In [ ]:
X = df[FEATURES]
y = df[TARGET]
groups = df['block_id']

spatial_cv = GroupKFold(n_splits=5)
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate(model, X, cv, groups=None):
    """Cross-validated R² and MAE (mean absolute error, in °C)."""
    scores = cross_validate(model, X, y, cv=cv, groups=groups,
                            scoring=['r2', 'neg_mean_absolute_error'])
    r2 = scores['test_r2'].mean()
    mae = -scores['test_neg_mean_absolute_error'].mean()
    return round(r2, 3), round(mae, 2)

## 4. Baseline: linear regression

A **baseline** is a simple model we try to beat. First, a linear regression using **only NDVI**, like your Earth Engine analysis.

- **R²**: share of the variation in temperature the model explains (1 = perfect, 0 = no better than always guessing the average).
- **MAE**: average error of the predictions, in °C.

In [ ]:
results = {}

r2, mae = evaluate(LinearRegression(), df[['ndvi']], spatial_cv, groups)
results['Linear regression, NDVI only'] = (r2, mae)
print(f'Linear regression, NDVI only -> R² = {r2}, MAE = {mae} °C')

### ✏️ YOUR TURN — Exercise 2: linear regression with all features

Evaluate a `LinearRegression()` using **all** the features (`X`) with the **spatial** cross-validation, and save the result in `results` with the name `'Linear regression, all features'`.

*Hint:* copy the cell above and change what is needed.

**Question:** how much does R² improve compared with NDVI only?

In [ ]:
# Exercise 2: linear regression with ALL features, spatial cross-validation
r2, mae = evaluate(LinearRegression(), X, spatial_cv, groups)
results['Linear regression, all features'] = (r2, mae)
print(f'Linear regression, all features -> R² = {r2}, MAE = {mae} °C')

*My answer (Exercise 2):*

Using only NDVI, the model explained 40% of the temperature differences in areas it had never seen (R² = 0.40), with a mean absolute error (MAE) of 1.54 °C. With all features, it explained 62% (R² = 0.62), and the error dropped to 1.23 °C. So R² increased by 0.22 and the error decreased by about 0.3 °C. This shows that other features, such as built-up cover and tree cover, also help predict temperature. As I expected, R² increased when more features were included.

## 5. Random forest

A **random forest** trains hundreds of decision trees, each on a slightly different part of the data, and averages their predictions. Unlike linear regression, it can learn **non-linear** relationships, like the pattern you found: temperature falls slowly with a little vegetation and quickly with dense vegetation.

`min_samples_leaf=5` stops each tree from memorising individual points.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                           n_jobs=-1, random_state=42)

### ✏️ YOUR TURN — Exercise 3: evaluate the random forest (two ways)

1. Evaluate `rf` with **spatial** cross-validation and save it in `results` as `'Random forest, spatial CV'`.
2. Evaluate `rf` with **random** cross-validation (`random_cv`, no `groups`) and save it as `'Random forest, random CV (optimistic)'`.

It may take a minute or two.

**Questions:**
1. Does the random forest beat the linear regression?
2. Which evaluation gives the better score, spatial or random? Explain **why** using what you read in section 3.

In [ ]:
# Exercise 3: random forest, evaluated two ways

# Test 1: SPATIAL cross-validation (honest), using the blocks
r2, mae = evaluate(rf, X, spatial_cv, groups)
results['Random forest, spatial CV'] = (r2, mae)
print(f'Random forest, spatial CV -> R² = {r2}, MAE = {mae} °C')

# Test 2: RANDOM cross-validation (optimistic), no blocks
r2, mae = evaluate(rf, X, random_cv)
results['Random forest, random CV (optimistic)'] = (r2, mae)
print(f'Random forest, random CV -> R² = {r2}, MAE = {mae} °C')

In [ ]:
pd.DataFrame(results, index=['R²', 'MAE (°C)']).T

*My answers (Exercise 3):*

1. Yes, the random forest performed slightly better than linear regression: R² of 0.65 compared to 0.62, and MAE of 1.17 °C instead of 1.23 °C. However, the improvement was small.
2. Random cross-validation gave a higher R² (0.75) than spatial cross-validation (0.65). This is because, with random cross-validation, the test pixels are close to the training pixels and very similar to them, so the model partly succeeds by recognising the neighbourhood rather than by truly learning the patterns. Spatial cross-validation is more realistic, because it tests the model in areas it has never seen. If I had used only random cross-validation, I would have overestimated the model's performance by about 10 percentage points.

## 6. Which features matter? Permutation importance

Idea: take a trained model, **shuffle one feature** in the test data (breaking its relationship with temperature) and measure how much the score drops. The bigger the drop, the more the model depended on that feature.

We train on some blocks and measure on blocks the model never saw.

In [ ]:
train_idx, test_idx = next(spatial_cv.split(X, y, groups))

rf.fit(X.iloc[train_idx], y.iloc[train_idx])

importance = permutation_importance(rf, X.iloc[test_idx], y.iloc[test_idx],
                                    n_repeats=10, random_state=42, n_jobs=-1)

imp = pd.Series(importance.importances_mean, index=FEATURES).sort_values()
imp.plot(kind='barh', figsize=(7, 4))
plt.xlabel('Drop in R² when the feature is shuffled')
plt.title('Permutation importance (random forest)')
plt.tight_layout()
plt.show()
imp.sort_values(ascending=False).round(3)

### ✏️ YOUR TURN — Exercise 4: interpret the importance

Answer in the cell below:

1. Which three features matter most?
2. Is NDVI still the most important once the model also knows tree cover, built-up cover and albedo? What could explain that?
3. *Careful:* when two features carry similar information (for example `ndvi` and `tree_cover_50m`), shuffling one of them may not hurt much, because the model can use the other. How does that affect the way you read this chart?

*My answers (Exercise 4):*

1. The three most important features for the model were: surrounding built-up cover (0.415), elevation (0.077) and surrounding tree cover (0.065).
2. No, NDVI ranked among the least important features (0.021). This is because the model also uses tree cover and built-up cover, which carry similar and even more precise information, since they separate trees from grass. So, without NDVI, the model still predicts almost as well.
3. When two features carry similar information, shuffling one of them does not hurt the model much, because it can rely on the other. So the chart shows which information is unique, not which factor matters most in the real world. Elevation ranks second because no other feature knows the height of the terrain. Vegetation is still very important, but its information is split across several features.

*Note:* importance was calculated using only one of the five spatial block splits. A future improvement is to repeat it for all five splits and average the results.

## 7. How does temperature change with tree cover? Partial dependence

A **partial dependence plot** shows how the predicted temperature changes when we vary one feature, averaging over the other features.

In [ ]:
rf.fit(X, y)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
PartialDependenceDisplay.from_estimator(
    rf, X, ['tree_cover_50m', 'ndvi', 'albedo'], ax=ax)
plt.tight_layout()
plt.show()

### ✏️ YOUR TURN — Exercise 5: read the curves

1. According to the model, about how many °C cooler is a place with **50% tree cover** within 50 m than a place with **0%**?
2. Is the curve for `ndvi` a straight line? Does it confirm the pattern "dense vegetation matters most" from your earlier analyses?
3. Why can't we say, from this model alone, that planting trees **will cause** exactly this cooling?

*My answers (Exercise 5):*

1. According to the model, a location with 50% surrounding tree cover is about 0.3 °C cooler than a location with no trees, assuming all other features stay the same. This effect is much smaller than the 6 °C difference found in previous analyses, because the chart changes only tree cover while keeping built-up cover constant. In reality, areas with more trees usually have less built-up cover. Also, 90% of the data points have less than one-third tree cover, so there is little data to support the final part of the curve.
2. No, the NDVI curve is almost flat, because the model relies little on NDVI: the information it provides is already captured by tree cover and built-up cover. However, the tree cover curve shows the sharpest drop when cover exceeds 85%, which suggests that dense vegetation provides the most cooling.
3. Because the model learned patterns in the data, not cause-and-effect relationships. Trees are often found in areas with fewer buildings and with other characteristics, which makes it hard for the model to separate their individual effects. Albedo is an example: the model predicted that more reflective surfaces would be warmer, which goes against physics. To know the real effect of planting trees, we would need an experiment, measuring temperature before and after planting.

## 8. Conclusions and limitations

**Conclusions:**

Using 7,956 pixels from the urban area of Santo André, the random forest predicted summer land surface temperature in areas it had never seen with R² = 0.65 and a mean absolute error of 1.17 °C (spatial cross-validation). This was slightly better than linear regression (R² = 0.62) and much better than using NDVI alone (R² = 0.40). Built-up cover was by far the most important feature, followed by elevation and tree cover: built-up areas are associated with higher temperatures, and areas with more trees tend to be cooler. We also found that random cross-validation would have overestimated performance (R² = 0.75), because test pixels close to training pixels are very similar to them. The model identifies patterns in the data, but it does not prove that trees cause lower temperatures, and it cannot measure the exact cooling effect of planting trees.

**Limitations:**

- The model predicts land surface temperature at about 10 a.m., not the air temperature people feel, and not afternoon or night-time heat.
- Most pixels have low tree cover, so predictions for densely vegetated areas are less certain.
- Features such as tree cover, built-up cover and NDVI carry similar information, which makes it hard to separate the effect of each one.
- Relationships in the data are associations, not cause and effect.
- Tree cover and built-up cover come from ESA WorldCover 2021, while the Landsat images are from 2023–2026, so some areas may have changed.
- Other factors are not included, such as roof materials, building height and shade, and wind.
- The model was tested only in Santo André; it may not work as well in other cities or other years.
-